# CEUNIA · Integración total para Google Colab

Este notebook reúne los repositorios disponibles en un espacio de trabajo trazable, inventaría los `.py` y `.ipynb`, calcula SHA-256 y crea un paquete consolidado **sin sobrescribir archivos homónimos ni ejecutar notebooks desconocidos automáticamente**. La integración funcional requiere después pruebas y resolución explícita de dependencias/conflictos.

Repositorios de origen: `CEUNIA-UNIFIED-SYSTEM`, `Ceunia`, `Ceunia.-Core`, `Ceunia_router`, `ceunia-riaas`.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, hashlib, shutil, zipfile, datetime, ast

# 1) Workspace limpio: los originales quedan separados por repositorio.
ROOT = Path('/content/CEUNIA_WORKSPACE')
ROOT.mkdir(parents=True, exist_ok=True)
REPOS = {
    'CEUNIA-UNIFIED-SYSTEM': 'https://github.com/matiasbision-dot/CEUNIA-UNIFIED-SYSTEM.git',
    'Ceunia': 'https://github.com/matiasbision-dot/Ceunia.git',
    'Ceunia.-Core': 'https://github.com/matiasbision-dot/Ceunia.-Core.git',
    'Ceunia_router': 'https://github.com/matiasbision-dot/Ceunia_router.git',
    'ceunia-riaas': 'https://github.com/matiasbision-dot/ceunia-riaas.git',
}
for name, url in REPOS.items():
    dest = ROOT / 'sources' / name
    if not (dest / '.git').exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists(): shutil.rmtree(dest)
        result = subprocess.run(['git','clone','--depth','1',url,str(dest)], text=True, capture_output=True)
        print(f'CLONE {name}:', 'OK' if result.returncode == 0 else 'ERROR')
        if result.returncode != 0: print(result.stderr[-1200:])
    else:
        print(f'EXISTENTE {name}: {dest}')

# 2) Inventario completo + hashes. No ejecuta el código fuente.
inventory = []
for repo_name in REPOS:
    repo_dir = ROOT / 'sources' / repo_name
    if not repo_dir.exists(): continue
    for p in repo_dir.rglob('*'):
        if not p.is_file() or '.git' in p.parts: continue
        rel = p.relative_to(repo_dir).as_posix()
        raw = p.read_bytes()
        row = {'repo':repo_name,'path':rel,'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest(),'type':p.suffix.lower()}
        if p.suffix.lower()=='.py':
            try: ast.parse(raw.decode('utf-8',errors='replace')); row['python_syntax']='OK'
            except SyntaxError as e: row['python_syntax']=f'ERROR line {e.lineno}: {e.msg}'
        elif p.suffix.lower()=='.ipynb':
            try:
                doc=json.loads(raw.decode('utf-8'))
                cells=doc.get('cells',[])
                row['notebook_cells']=len(cells)
                row['code_cells']=sum(c.get('cell_type')=='code' for c in cells)
                row['notebook_json']='OK'
            except Exception as e: row['notebook_json']=f'ERROR: {e}'
        inventory.append(row)

manifest = {'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'policy':'source files preserved; no automatic execution of imported notebooks','repositories':REPOS,'files':inventory}
(ROOT/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('\nARCHIVOS:',len(inventory),'| PY:',sum(x['type']=='.py' for x in inventory),'| NOTEBOOKS:',sum(x['type']=='.ipynb' for x in inventory))
print('Manifest:', ROOT/'manifest.json')

# 3) Capa consolidada: copia todos los archivos con prefijo de origen, evitando colisiones.
# Los originales permanecen intactos en sources/.
bundle = ROOT/'consolidated_sources'
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
for row in inventory:
    src = ROOT/'sources'/row['repo']/row['path']
    dst = bundle/row['repo']/row['path']
    dst.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(src,dst)

# 4) Informe de riesgos estático: detecta señales para revisión manual, no prueba que sean vulnerabilidades.
signals = []
needles = ['eval(', 'exec(', 'shell=True', 'os.system(', 'subprocess.Popen(', 'google.colab.auth', 'userdata.get(', 'api_key', 'secret_key', 'password =']
for row in inventory:
    if row['type'] not in ('.py','.ipynb'): continue
    p = ROOT/'sources'/row['repo']/row['path']
    try: txt=p.read_text(encoding='utf-8',errors='replace')
    except Exception: continue
    for line_no,line in enumerate(txt.splitlines(),1):
        low=line.lower()
        for needle in needles:
            if needle.lower() in low:
                signals.append({'repo':row['repo'],'path':row['path'],'line':line_no,'pattern':needle})
(ROOT/'static_review_signals.json').write_text(json.dumps(signals,indent=2,ensure_ascii=False),encoding='utf-8')
print('Señales para revisión manual:',len(signals))

# 5) Paquete descargable con fuentes, manifest e informe. Excluye .git para reducir tamaño.
zip_path = Path('/content/CEUNIA_INTEGRACION_TOTAL_COLAB.zip')
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for base in [ROOT/'sources', bundle, ROOT/'manifest.json', ROOT/'static_review_signals.json']:
        if base.is_file(): z.write(base,base.relative_to(ROOT))
        elif base.exists():
            for p in base.rglob('*'):
                if p.is_file() and '.git' not in p.parts: z.write(p,p.relative_to(ROOT))
print('PAQUETE:',zip_path)

# 6) Tabla resumida para auditar el alcance de la integración.
from collections import Counter
print('\nPOR REPOSITORIO:')
for name in REPOS:
    rows=[r for r in inventory if r['repo']==name]
    print(f'- {name}: {len(rows)} archivos; {sum(r["type"]==".py" for r in rows)} Python; {sum(r["type"]==".ipynb" for r in rows)} notebooks')

print('\nSIGUIENTE FASE: resolver dependencias, nombres duplicados, APIs incompatibles y ejecutar pruebas por notebook en entorno aislado. No ejecutar todas las celdas a ciegas.')


## Uso
1. Abrí este archivo con Google Colab.
2. Ejecutá la única celda de integración.
3. Revisá el inventario y las señales de revisión.
4. Descargá `CEUNIA_INTEGRACION_TOTAL_COLAB.zip` desde el panel de archivos de Colab.

**Límite importante:** reunir todos los archivos con trazabilidad no significa que todos los experimentos estén integrados semánticamente. Los notebooks pueden depender de secretos, datos externos, versiones antiguas o estados de ejecución previos. La ejecución masiva automática sería insegura y podría alterar resultados históricos.
